# LeetCode 212 — Word Search II (Hard): nhập môn DFS trên lưới + Trie bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findWords(self, board: List[List[str]], words: List[str]) -> List[str]:

```

Bài này là sự kết hợp của hai kỹ thuật rất hay gặp trong phỏng vấn:

- **DFS / backtracking trên lưới** (giống bài #79 Word Search): đi từ ô này sang ô kề, không được dùng lại ô đã đi.
- **Trie** (cây tiền tố): cấu trúc dữ liệu giúp hỏi nhanh "chuỗi đang ghép có còn là **đầu** của từ nào không?".

---

## Sau notebook này, bạn sẽ nắm được

1. Cách biểu diễn lưới 2D, duyệt **4 hướng** và kiểm tra "còn trong lưới".
2. Đệ quy, DFS trên lưới, và sự khác nhau giữa **visited vĩnh viễn** (flood fill) và **visited tạm thời** (backtracking, phải trả lại).
3. Giải bài nền #79 Word Search (tìm **một** từ) rồi dùng nó làm brute force cho bài này.
4. Đảo góc nhìn: thay vì "từ → lưới", ta đi "lưới → từ" và **cắt nhánh** bằng tập tiền tố.
5. **Trie từ căn bản**: `insert`, `search`, `startsWith`, vẽ cây Trie dạng ASCII.
6. Cách chính: **Trie + DFS một lần trên lưới**, rồi 3 tối ưu: lưu `word` tại node cuối, xóa `word` khi đã tìm thấy, **tỉa node lá** (pruning) — có đo số bước DFS trước/sau.
7. Lỗi hay gặp, mẹo nhận dạng, biến thể (#79, #208, #211...) và 6 bài tập có đáp án ở cuối.

## 1. Đề bài

Cho một lưới ký tự `board` kích thước `m x n` và một danh sách chuỗi `words`, hãy trả về **tất cả các từ** trong `words` xuất hiện trên lưới.

Mỗi từ phải được ghép từ các chữ cái của những ô **kề nhau liên tiếp**, trong đó hai ô kề nhau là hai ô cạnh nhau theo **chiều ngang hoặc chiều dọc**. **Cùng một ô chữ không được dùng quá một lần trong một từ.**

**Example 1**

```text
Input: board = [["o","a","a","n"],["e","t","a","e"],["i","h","k","r"],["i","f","l","v"]], words = ["oath","pea","eat","rain"]
Output: ["eat","oath"]
```

**Example 2**

```text
Input: board = [["a","b"],["c","d"]], words = ["abcb"]
Output: []
```

**Constraints**

- `m == board.length`
- `n == board[i].length`
- `1 <= m, n <= 12`
- `board[i][j]` là chữ cái tiếng Anh viết thường.
- `1 <= words.length <= 3 * 10^4`
- `1 <= words[i].length <= 10`
- `words[i]` chỉ gồm chữ cái tiếng Anh viết thường.
- Tất cả các chuỗi trong `words` là **khác nhau** (unique).

Ghi chú: đề không ràng buộc thứ tự kết quả; trình chấm LeetCode chấp nhận mọi thứ tự (Example 1 có thể trả `["oath","eat"]`). Vì vậy trong notebook ta luôn so sánh sau khi `sorted(...)`.

Quy ước: ô `(r, c)` là hàng `r` (row, đếm từ 0, từ trên xuống) và cột `c` (column, đếm từ 0, từ trái sang). Ta gọi:

- `m`, `n`: số hàng, số cột;
- `W`: số từ trong `words`;
- `L`: độ dài từ dài nhất (`L <= 10`);
- `S`: tổng số ký tự của mọi từ (`S <= 3 * 10^4 * 10 = 3 * 10^5`).

In [1]:
from typing import List
import copy

board1 = [["o","a","a","n"],
          ["e","t","a","e"],
          ["i","h","k","r"],
          ["i","f","l","v"]]
words1 = ["oath", "pea", "eat", "rain"]

board2 = [["a","b"],
          ["c","d"]]
words2 = ["abcb"]

print("m =", len(board1), ", n =", len(board1[0]))
print("Hàng 0:", board1[0])
print("Ô (1, 1) =", board1[1][1])

m = 4 , n = 4
Hàng 0: ['o', 'a', 'a', 'n']
Ô (1, 1) = t


### Nhìn tận mắt các từ trong Example 1

Hàm nhỏ `show_board(board, path)` in lưới và đánh **số thứ tự** lên các ô thuộc đường đi `path` (một list tọa độ). Ô không thuộc đường đi in kèm dấu cách.

- `"oath"`: `(0,0) o → (0,1) a → (1,1) t → (2,1) h`. Mỗi bước đi sang ô kề ngang/dọc. ✔
- `"eat"`: `(1,3) e → (1,2) a → (1,1) t`. ✔
- `"pea"`: lưới không có chữ `p` nào → không thể. ✘
- `"rain"`: có `r` ở `(2,3)`, kề với nó có `a`? Các ô kề `(2,3)` là `(1,3) e`, `(3,3) v`, `(2,2) k` → không có `a`. ✘

Example 2: `"abcb"` cần đi `a → b → c → b`. Từ `a(0,0)` sang `b(0,1)`, nhưng `b(0,1)` không kề `c(1,0)` (chúng chéo nhau). Kể cả nếu đi được thì chữ `b` cuối phải dùng **lại** ô `(0,1)` — bị cấm.

In [2]:
def show_board(board, path=()):
    order = {cell: k for k, cell in enumerate(path, 1)}   # ô -> số thứ tự trên đường đi
    n = len(board[0])
    print("    " + " ".join(f"c{c}" for c in range(n)))
    for r, row in enumerate(board):
        items = [f"{ch}{order[(r, c)]}" if (r, c) in order else f"{ch} "
                 for c, ch in enumerate(row)]
        print(f"r{r}  " + " ".join(items))

print('"oath":')
show_board(board1, [(0, 0), (0, 1), (1, 1), (2, 1)])
print('\n"eat":')
show_board(board1, [(1, 3), (1, 2), (1, 1)])

"oath":
    c0 c1 c2 c3
r0  o1 a2 a  n 
r1  e  t3 a  e 
r2  i  h4 k  r 
r3  i  f  l  v 

"eat":
    c0 c1 c2 c3
r0  o  a  a  n 
r1  e  t3 a2 e1
r2  i  h  k  r 
r3  i  f  l  v 


## 2. Nền tảng 1 — Lưới 2D, chỉ số và 4 hướng đi

Trong Python, lưới là **list của các list**: `board[r]` là cả hàng `r`, `board[r][c]` là một ô.

- Số hàng: `m = len(board)`; số cột: `n = len(board[0])`.
- Ô `(r, c)` có tối đa **4 ô kề**: lên `(r-1, c)`, xuống `(r+1, c)`, trái `(r, c-1)`, phải `(r, c+1)`.

Thay vì viết 4 đoạn `if` gần giống nhau, ta dùng một list **độ lệch** (offset):

```python
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải
for dr, dc in DIRS:
    nr, nc = r + dr, c + dc
```

Ô mới `(nr, nc)` có thể **rơi ra ngoài** lưới (ví dụ ô góc `(0,0)` đi lên thành `(-1, 0)`). Phải kiểm tra:

```python
0 <= nr < m and 0 <= nc < n
```

Đừng quên: trong Python `board[-1]` **không báo lỗi** mà trả về hàng cuối! Nếu quên kiểm tra biên, chương trình chạy "êm" nhưng sai.

In [3]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải

def neighbors(board, r, c):
    m, n = len(board), len(board[0])
    result = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:          # còn trong lưới
            result.append((nr, nc))
    return result

print("Ô góc (0,0):", neighbors(board1, 0, 0))
print("Ô giữa (1,1):", neighbors(board1, 1, 1))
print("Ô cạnh (3,2):", neighbors(board1, 3, 2))
print("Cẩn thận: board1[-1] =", board1[-1], "(không lỗi nhưng là hàng cuối!)")

Ô góc (0,0): [(1, 0), (0, 1)]
Ô giữa (1,1): [(0, 1), (2, 1), (1, 0), (1, 2)]
Ô cạnh (3,2): [(2, 2), (3, 1), (3, 3)]
Cẩn thận: board1[-1] = ['i', 'f', 'l', 'v'] (không lỗi nhưng là hàng cuối!)


## 3. Nền tảng 2 — Đệ quy (recursion) và DFS trên lưới

### 3.1 Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó với bài toán **nhỏ hơn**, và có **điểm dừng** (base case) để không gọi mãi.

Mỗi lần gọi, Python đặt một "khung" mới lên **call stack** (ngăn xếp lời gọi). Khi lời gọi con xong, Python quay về đúng chỗ lời gọi cha đang đứng. Cell dưới đây in thụt lề theo độ sâu để bạn thấy "đi xuống" rồi "quay về".

In [4]:
def countdown(k, depth=0):
    pad = "  " * depth
    if k == 0:                       # điểm dừng
        print(pad + "k = 0 -> dừng, bắt đầu quay về")
        return
    print(pad + f"vào countdown({k})")
    countdown(k - 1, depth + 1)      # bài toán nhỏ hơn
    print(pad + f"quay về countdown({k})")

countdown(3)

vào countdown(3)
  vào countdown(2)
    vào countdown(1)
      k = 0 -> dừng, bắt đầu quay về
    quay về countdown(1)
  quay về countdown(2)
quay về countdown(3)


### 3.2 DFS (Depth-First Search) trên lưới — ví dụ flood fill

**DFS** (tìm kiếm theo chiều sâu) nghĩa là: từ một ô, đi **sâu hết mức** theo một hướng, bí thì **quay lui** về ô trước và thử hướng khác. Viết bằng đệ quy rất tự nhiên:

```python
def dfs(r, c):
    đánh dấu (r, c) đã thăm
    for mỗi ô kề (nr, nc) hợp lệ và chưa thăm:
        dfs(nr, nc)
```

Ví dụ mini **flood fill**: đếm số ô chữ `'a'` liên thông với ô `(0,0)` trong lưới bên dưới. Ở đây `visited` là **vĩnh viễn**: một ô đã đếm thì không bao giờ cần đếm lại.

Ta dùng `set` (tập hợp) để lưu ô đã thăm: `visited.add((r, c))` thêm vào, `(r, c) in visited` kiểm tra có chưa — trung bình `O(1)`.

In [5]:
grid = [["a", "a", "b"],
        ["b", "a", "a"],
        ["a", "b", "a"]]

def flood_count(grid, r, c, visited):
    visited.add((r, c))                       # đánh dấu VĨNH VIỄN
    total = 1
    for nr, nc in neighbors(grid, r, c):
        if grid[nr][nc] == "a" and (nr, nc) not in visited:
            total += flood_count(grid, nr, nc, visited)
    return total

seen = set()
print("Số ô 'a' liên thông với (0,0):", flood_count(grid, 0, 0, seen))
print("Các ô đã thăm:", sorted(seen))

Số ô 'a' liên thông với (0,0): 5
Các ô đã thăm: [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)]


Kết quả `5`: các ô `(0,0) (0,1) (1,1) (1,2) (2,2)`. Ô `(2,0)` cũng là `'a'` nhưng bị các ô `'b'` cô lập nên không được đếm.

Nhưng bài Word Search **khác** flood fill ở một điểm quan trọng: ta không đếm ô, ta đi theo **một đường** (path) để ghép chữ. Một ô bị cấm dùng lại **trong cùng một đường**, nhưng **đường khác** vẫn được dùng nó. Đó là lúc cần backtracking.

## 4. Nền tảng 3 — Backtracking: đánh dấu visited **tạm thời** rồi trả lại

**Backtracking** (quay lui) theo khuôn mẫu 3 bước:

1. **Chọn**: đưa ô `(r, c)` vào đường đi và đánh dấu "đang dùng".
2. **Đi tiếp**: gọi đệ quy sang các ô kề.
3. **Bỏ chọn**: khi quay về, **gỡ dấu** để đường đi khác còn dùng được ô này.

Mẹo kinh điển trên lưới chữ: thay vì `set` visited, ta **ghi đè tạm** ô bằng ký tự `'#'` (không phải chữ cái nào), rồi trả lại chữ cũ:

```python
ch = board[r][c]
board[r][c] = "#"        # chọn: ô đang nằm trên đường đi
... đệ quy ...
board[r][c] = ch         # bỏ chọn: trả lại chữ cũ
```

Vì `'#'` không bằng chữ nào, ô đang dùng tự động "không khớp" — không cần thêm điều kiện.

Ví dụ mini: liệt kê **mọi chuỗi** ghép được từ ô `(0,0)` trên lưới `2 x 2` của Example 2, độ dài đúng 3 và đúng 4. Tính tay trước:

- từ `a(0,0)` sang `b(0,1)` hoặc `c(1,0)`;
- từ `b(0,1)`: kề `a` (đang dùng) và `d(1,1)` → `"abd"`; từ `c(1,0)`: kề `a` (đang dùng) và `d` → `"acd"`;
- thêm một bước: `"abd" → c` = `"abdc"`, `"acd" → b` = `"acdb"`.

In [6]:
def all_strings_from(board, r, c, length):
    m, n = len(board), len(board[0])
    out = []

    def dfs(r, c, s):
        s = s + board[r][c]
        if len(s) == length:
            out.append(s)
            return
        ch = board[r][c]
        board[r][c] = "#"                         # chọn
        for nr, nc in neighbors(board, r, c):
            if board[nr][nc] != "#":              # chưa nằm trên đường đi
                dfs(nr, nc, s)                    # đi tiếp
        board[r][c] = ch                          # bỏ chọn

    dfs(r, c, "")
    return out

b = copy.deepcopy(board2)
print("Độ dài 3:", all_strings_from(b, 0, 0, 3))
print("Độ dài 4:", all_strings_from(b, 0, 0, 4))
print("Lưới sau khi chạy (phải nguyên vẹn):", b)

Độ dài 3: ['acd', 'abd']
Độ dài 4: ['acdb', 'abdc']
Lưới sau khi chạy (phải nguyên vẹn): [['a', 'b'], ['c', 'd']]


Hai điều cần thấy:

- Chuỗi `"abdb"` **không** xuất hiện: ô `b(0,1)` đang mang dấu `'#'` nên không dùng lại được.
- Lưới sau khi chạy **giống hệt ban đầu**: mỗi lần "chọn" đều có một lần "bỏ chọn" tương ứng. Nếu quên trả lại, lưới bị "thủng lỗ `#`" và các lần tìm sau sai (xem Lỗi 1 ở mục 17).

Tại sao dùng `copy.deepcopy(board2)`? Vì hàm sửa tạm lưới; dùng bản sao giúp các cell demo không ảnh hưởng nhau kể cả khi có lỗi giữa chừng. (`board2[:]` chỉ copy **lớp ngoài**, các hàng bên trong vẫn dùng chung.)

## 5. Bài nền #79 Word Search — tìm **một** từ trên lưới

Bài #79 hỏi: lưới có chứa **một** từ `word` không? (trả `True/False`). Đây chính là "Word Search I".

Ý tưởng: thử mọi ô làm điểm xuất phát. Hàm `dfs(r, c, i)` trả lời câu hỏi:

> "Bắt đầu tại ô `(r, c)`, có ghép được phần `word[i:]` không?"

1. Nếu `board[r][c] != word[i]` → `False` (sai chữ, hoặc ô đang là `'#'`).
2. Nếu `i == len(word) - 1` → đã khớp chữ cuối → `True`.
3. Ngược lại: đánh dấu `'#'`, thử 4 ô kề với `i + 1`; trả lại chữ cũ trước khi `return`.

Ví dụ: `word = "eat"`, xuất phát `(1,3)`: `e` khớp `word[0]` → sang `(1,2)` `a` khớp `word[1]` → sang `(1,1)` `t` khớp `word[2]` và là chữ cuối → `True`.

In [7]:
def exist(board, word, stats=None):
    m, n = len(board), len(board[0])

    def dfs(r, c, i):
        if stats is not None:
            stats["dfs"] += 1                     # đếm số lần gọi dfs (để so sánh sau)
        if board[r][c] != word[i]:
            return False
        if i == len(word) - 1:
            return True
        ch = board[r][c]
        board[r][c] = "#"                         # chọn
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and dfs(nr, nc, i + 1):
                board[r][c] = ch                  # nhớ trả lại TRƯỚC khi return True
                return True
        board[r][c] = ch                          # bỏ chọn
        return False

    return any(dfs(r, c, 0) for r in range(m) for c in range(n))

for w in words1:
    print(f"{w!r:7} có trên board1?", exist(copy.deepcopy(board1), w))
print("'abcb'  có trên board2?", exist(copy.deepcopy(board2), "abcb"))

'oath'  có trên board1? True
'pea'   có trên board1? False
'eat'   có trên board1? True
'rain'  có trên board1? False
'abcb'  có trên board2? False


### Trace #79 với `"abcb"` trên lưới 2x2 — thấy dấu `'#'` chặn việc dùng lại ô

Hàm `trace_exist` in mỗi lời gọi `dfs` (thụt lề theo độ sâu), chữ cần khớp `word[i]` và kết quả. Chỉ ô `(0,0)` có chữ `a` nên chỉ nó đi được sâu; các ô xuất phát khác thất bại ngay ở bước 1.

In [8]:
def trace_exist(board, word):
    m, n = len(board), len(board[0])

    def dfs(r, c, i, depth):
        pad = "  " * depth
        if board[r][c] != word[i]:
            print(f"{pad}({r},{c})='{board[r][c]}' cần '{word[i]}' -> sai, quay lui")
            return False
        if i == len(word) - 1:
            print(f"{pad}({r},{c})='{board[r][c]}' khớp chữ cuối -> True")
            return True
        print(f"{pad}({r},{c})='{board[r][c]}' khớp word[{i}], đánh dấu '#'")
        ch, board[r][c] = board[r][c], "#"
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and dfs(nr, nc, i + 1, depth + 1):
                board[r][c] = ch
                return True
        board[r][c] = ch
        print(f"{pad}trả lại ({r},{c})='{ch}'")
        return False

    for r in range(m):
        for c in range(n):
            print(f"-- xuất phát ({r},{c})")
            if dfs(r, c, 0, 1):
                return True
    return False

print("Kết quả:", trace_exist(copy.deepcopy(board2), "abcb"))

-- xuất phát (0,0)
  (0,0)='a' khớp word[0], đánh dấu '#'
    (1,0)='c' cần 'b' -> sai, quay lui
    (0,1)='b' khớp word[1], đánh dấu '#'
      (1,1)='d' cần 'c' -> sai, quay lui
      (0,0)='#' cần 'c' -> sai, quay lui
    trả lại (0,1)='b'
  trả lại (0,0)='a'
-- xuất phát (0,1)
  (0,1)='b' cần 'a' -> sai, quay lui
-- xuất phát (1,0)
  (1,0)='c' cần 'a' -> sai, quay lui
-- xuất phát (1,1)
  (1,1)='d' cần 'a' -> sai, quay lui
Kết quả: False


Đọc trace: từ `a(0,0)` ta sang được `b(0,1)` (khớp `word[1]`), nhưng từ `b(0,1)` cần chữ `'c'`: ô kề `(1,1)` là `d` (sai) và ô kề `(0,0)` đang là `'#'` (sai). Không có đường nào → `False`, đúng như Example 2.

**Độ phức tạp #79** với một từ độ dài `L`: mỗi ô xuất phát, bước đầu có tối đa 4 hướng, các bước sau tối đa 3 hướng (không quay lại ô vừa đi) → `O(m·n·4·3^(L-1))`, thường viết gọn `O(m·n·4^L)`. Bộ nhớ `O(L)` cho call stack.

## 6. Cách 1 — Brute force: chạy Word Search I cho **từng từ**

Cách người mới nghĩ ra đầu tiên: "Mình đã có `exist(board, word)`, vậy cứ hỏi lần lượt từng từ!"

```python
return [w for w in words if exist(board, w)]
```

Đúng, ngắn, dễ hiểu. Hàm dưới nhận thêm `stats` để **đếm tổng số lần gọi `dfs`** — đơn vị "bước DFS" ta sẽ dùng để so sánh các cách.

In [9]:
def word_search_each(board, words, stats=None):
    return [w for w in words if exist(board, w, stats)]

st = {"dfs": 0}
print("Example 1:", sorted(word_search_each(copy.deepcopy(board1), words1, st)), "| bước DFS:", st["dfs"])
st = {"dfs": 0}
print("Example 2:", sorted(word_search_each(copy.deepcopy(board2), words2, st)), "| bước DFS:", st["dfs"])

Example 1: ['eat', 'oath'] | bước DFS: 58
Example 2: [] | bước DFS: 8


### Nút thắt của cách 1

**Thời gian** `O(W · m·n·4·3^(L-1))` (gọn: `O(W · m·n·4^L)`). Với constraints: `W = 3·10^4`, `m·n = 144`, `L = 10` → cận trên khoảng `3·10^4 · 144 · 4 · 3^9 ≈ 3,4 · 10^11` bước. Không thể chạy kịp.

Nhưng vấn đề sâu hơn là **công việc bị lặp**. Giả sử `words = ["oat", "oath", "oaths"]`: cả 3 lần gọi `exist` đều đi lại **đúng một đoạn đường** `o → a → t` trên lưới. Các từ có chung **tiền tố** (prefix: phần đầu của chuỗi, ví dụ `"oa"` là tiền tố của `"oath"`) nhưng cách 1 không tận dụng điều đó.

Ngoài ra, mỗi từ đều phải quét **hết 144 ô** xuất phát, kể cả khi từ đó bắt đầu bằng chữ không có trên lưới.

Câu hỏi dẫn đường: **có thể đi trên lưới chỉ một lần, và kiểm tra tất cả các từ cùng lúc không?**

## 7. Cách 2 — Đảo góc nhìn: DFS từ mỗi ô, sinh chuỗi, tra `set`

Thay vì "từ → tìm trên lưới", ta làm "**lưới → sinh chuỗi → tra xem có phải từ không**":

- DFS từ mỗi ô, ghép dần chuỗi `s` dọc đường đi (dùng đúng khuôn backtracking ở mục 4).
- Mỗi khi có `s`, tra `s in word_set` (tập các từ) — trung bình `O(len(s))` để băm chuỗi.
- Vì mọi từ dài `<= L`, chỉ cần sinh chuỗi dài `<= L`.

### 7.1 Phiên bản chưa cắt nhánh: sinh **mọi** chuỗi độ dài `<= L`

Ta dùng `set found` để mỗi từ chỉ ghi nhận một lần (một từ có thể ghép được bằng nhiều đường khác nhau). Hàm này cũng là **brute force tuyệt đối** — dùng để đối chiếu chéo ở cuối.

In [10]:
def find_words_all_paths(board, words, stats=None):
    word_set = set(words)
    L = max(len(w) for w in words)
    m, n = len(board), len(board[0])
    found = set()

    def dfs(r, c, s):
        if stats is not None:
            stats["dfs"] += 1
        s = s + board[r][c]
        if s in word_set:
            found.add(s)
        if len(s) == L:                            # đủ dài nhất có thể, dừng
            return
        ch = board[r][c]
        board[r][c] = "#"
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and board[nr][nc] != "#":
                dfs(nr, nc, s)
        board[r][c] = ch

    for r in range(m):
        for c in range(n):
            dfs(r, c, "")
    return list(found)

st = {"dfs": 0}
print("Example 1:", sorted(find_words_all_paths(copy.deepcopy(board1), words1, st)), "| bước DFS:", st["dfs"])

Example 1: ['eat', 'oath'] | bước DFS: 400


Tốn **nhiều bước hơn** cả cách 1 trên Example 1! Lý do: ta sinh cả những chuỗi vô vọng như `"ie"`, `"ihk..."` — chẳng từ nào bắt đầu như vậy, nhưng DFS vẫn đi tiếp tới độ dài 4.

Với lưới `12 x 12` và `L = 10`, số đường đi có thể lên tới cỡ `144 · 4 · 3^9 ≈ 1,1 · 10^7` — mỗi bước lại tạo chuỗi mới. Cần **cắt nhánh** (pruning): dừng ngay khi chuỗi đang ghép không thể trở thành từ nào.

### 7.2 Mấu chốt: tập tiền tố `prefixes`

Nếu `s` **không phải tiền tố** của từ nào thì đi tiếp cũng vô ích → dừng. Ta tính trước tập mọi tiền tố của mọi từ:

`"oath"` → `"o"`, `"oa"`, `"oat"`, `"oath"`.

Biểu thức `w[:k]` (slicing) lấy `k` ký tự đầu của `w`.

In [11]:
w = "oath"
print([w[:k] for k in range(1, len(w) + 1)])

prefixes = {w[:k] for w in words1 for k in range(1, len(w) + 1)}
print("Số tiền tố:", len(prefixes))
print(sorted(prefixes))
print("'oa' in prefixes ->", "oa" in prefixes, "| 'ie' in prefixes ->", "ie" in prefixes)

['o', 'oa', 'oat', 'oath']
Số tiền tố: 14
['e', 'ea', 'eat', 'o', 'oa', 'oat', 'oath', 'p', 'pe', 'pea', 'r', 'ra', 'rai', 'rain']
'oa' in prefixes -> True | 'ie' in prefixes -> False


In [12]:
def find_words_prefix_set(board, words, stats=None):
    word_set = set(words)
    prefixes = {w[:k] for w in words for k in range(1, len(w) + 1)}
    m, n = len(board), len(board[0])
    found = set()

    def dfs(r, c, s):
        if stats is not None:
            stats["dfs"] += 1
        s = s + board[r][c]
        if s not in prefixes:                      # CẮT NHÁNH (kể cả khi ô là '#')
            return
        if s in word_set:
            found.add(s)
        ch = board[r][c]
        board[r][c] = "#"
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                dfs(nr, nc, s)                     # ô '#' cho s + '#' -> bị cắt ở trên
        board[r][c] = ch

    for r in range(m):
        for c in range(n):
            dfs(r, c, "")
    return list(found)

for name, f in [("chưa cắt nhánh", find_words_all_paths), ("cắt bằng prefixes", find_words_prefix_set)]:
    st = {"dfs": 0}
    res = sorted(f(copy.deepcopy(board1), words1, st))
    print(f"{name:18}: {res} | bước DFS: {st['dfs']}")

chưa cắt nhánh    : ['eat', 'oath'] | bước DFS: 400
cắt bằng prefixes : ['eat', 'oath'] | bước DFS: 46


### Nút thắt còn lại của cách 2

Số bước DFS giảm từ 400 xuống 46. Nhưng **mỗi bước** vẫn tốn kém:

1. `s = s + board[r][c]` tạo **chuỗi mới** dài tới `L` ký tự → `O(L)`.
2. `s in prefixes` phải **băm** (hash) cả chuỗi `s` → thêm `O(L)`.
3. Bộ nhớ: tập `prefixes` lưu riêng từng tiền tố. Với `W = 3·10^4` từ dài 10, có tới `3·10^5` tiền tố, mỗi cái dài tới 10 ký tự → cỡ `O(S·L)` ký tự, dù các từ chia sẻ phần đầu rất nhiều.

Điều ta thực sự cần là: đang ở tiền tố `"oa"`, thêm chữ `'t'` → **nhảy ngay** sang trạng thái `"oat"` trong `O(1)`, không băm lại cả chuỗi. Và các từ có chung tiền tố thì chỉ lưu tiền tố đó **một lần**. Cấu trúc làm được cả hai việc: **Trie**.

## 8. Nền tảng 4 — Trie (cây tiền tố) từ căn bản

**Trie** (đọc là "try", từ chữ re*trie*val) là một cây mà:

- mỗi **node** đại diện cho **một tiền tố**; node gốc `root` là chuỗi rỗng `""`;
- mỗi **cạnh** mang một ký tự: đi từ node `"oa"` theo cạnh `'t'` tới node `"oat"`;
- node nào là **trọn một từ** thì được đánh dấu (ở đây vẽ `★`).

Ví dụ Trie của `["oat", "oath", "eat", "ear"]`:

```text
(root)
├── o
│   └── a
│       └── t ★ "oat"
│           └── h ★ "oath"
└── e
    └── a
        ├── t ★ "eat"
        └── r ★ "ear"
```

Quan sát:

- `"oat"` và `"oath"` **dùng chung** 3 node `o → a → t`; `"eat"` và `"ear"` dùng chung `e → a`. Tổng chỉ 8 node (không tính root), trong khi tổng số ký tự là 3 + 4 + 3 + 3 = 13.
- Node `t` của `"oat"` vừa là từ (`★`) vừa có con `h` — đánh dấu từ **không** có nghĩa là node lá.
- Muốn biết `"ea"` có phải tiền tố của từ nào không: đi `root → e → a`, đi được → có.

### Mỗi node lưu gì?

Một `dict` `children` (ký tự → node con) và một cờ `is_end` (node này có phải trọn một từ). Nhắc nhanh về `dict`:

In [13]:
children = {}
children["a"] = "node A"            # thêm cặp key -> value
print(children)                     # {'a': 'node A'}
print("a" in children, "b" in children)
print(children.get("a"), children.get("b"))   # .get trả None nếu không có key, KHÔNG báo lỗi

x = children.setdefault("b", "node B mới")    # chưa có 'b' -> tạo và trả về
y = children.setdefault("a", "không dùng")    # đã có 'a' -> giữ nguyên, trả cái cũ
print(x, "|", y, "|", children)

{'a': 'node A'}
True False
node A None
node B mới | node A | {'a': 'node A', 'b': 'node B mới'}


### Ba thao tác của Trie (đúng tên hàm bài #208 Implement Trie)

- `insert(word)`: đi từ `root` theo từng ký tự; thiếu node con thì **tạo mới**; tới cuối đặt `is_end = True`.
- `search(word)`: đi theo từng ký tự; gãy giữa chừng → `False`; đi hết → trả `node.is_end` (phải là **trọn từ**).
- `startsWith(prefix)`: giống `search` nhưng đi hết là `True` luôn (chỉ cần **là tiền tố**).

Mỗi thao tác tốn `O(độ dài chuỗi)` — **không phụ thuộc** số từ đã chèn.

Hàm `_walk(s)` dùng chung: trả về node ứng với tiền tố `s`, hoặc `None` nếu không đi được.

In [14]:
class TrieNode:
    def __init__(self):
        self.children = {}      # ký tự -> TrieNode
        self.is_end = False     # True nếu đường từ root tới đây là trọn một từ

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()    # thiếu thì tạo
            node = node.children[ch]              # đi xuống
        node.is_end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word: str) -> bool:
        node = self._walk(word)
        return node is not None and node.is_end

    def startsWith(self, prefix: str) -> bool:
        return self._walk(prefix) is not None

t = Trie()
for w in ["oat", "oath", "eat", "ear"]:
    t.insert(w)
print("search('oat')     =", t.search("oat"))
print("search('oa')      =", t.search("oa"), " <- 'oa' chỉ là tiền tố, không phải từ")
print("startsWith('oa')  =", t.startsWith("oa"))
print("startsWith('ex')  =", t.startsWith("ex"))
print("search('oaths')   =", t.search("oaths"))

search('oat')     = True
search('oa')      = False  <- 'oa' chỉ là tiền tố, không phải từ
startsWith('oa')  = True
startsWith('ex')  = False
search('oaths')   = False


### Vẽ cây Trie dạng ASCII

Hàm `print_trie` duyệt cây (cũng là một DFS, trên Trie thay vì trên lưới) và in giống hình ở trên. Nó hoạt động với cả node có `is_end` (mục này) lẫn node có `word` (mục 10). Dùng `├──` cho con chưa phải cuối, `└──` cho con cuối cùng, và `│   ` để nối dọc.

In [15]:
def print_trie(node, prefix="", path=""):
    if prefix == "":
        print("(root)")
    items = list(node.children.items())
    for k, (ch, child) in enumerate(items):
        last = (k == len(items) - 1)
        word = getattr(child, "word", None)
        if word is None and getattr(child, "is_end", False):
            word = path + ch
        mark = f' ★ "{word}"' if word is not None else ""
        print(prefix + ("└── " if last else "├── ") + ch + mark)
        print_trie(child, prefix + ("    " if last else "│   "), path + ch)

print_trie(t.root)

(root)
├── o
│   └── a
│       └── t ★ "oat"
│           └── h ★ "oath"
└── e
    └── a
        ├── t ★ "eat"
        └── r ★ "ear"


### Trace `insert`: node nào tạo mới, node nào dùng lại?

Chèn lần lượt `"oat"`, `"oath"`, `"eat"`, `"ear"` vào Trie rỗng. Để ý `"oath"` chỉ tạo **1** node mới (`h`), `"ear"` chỉ tạo **1** node mới (`r`) — đó là sự chia sẻ tiền tố.

In [16]:
def trace_insert(words):
    root = TrieNode()
    total = 0
    for w in words:
        node, steps = root, []
        for ch in w:
            if ch in node.children:
                steps.append(f"{ch}:dùng lại")
            else:
                node.children[ch] = TrieNode()
                steps.append(f"{ch}:TẠO MỚI")
                total += 1
            node = node.children[ch]
        node.is_end = True
        print(f"insert({w!r:7}) -> " + ", ".join(steps))
    print("Tổng số node (không tính root):", total)
    return root

_ = trace_insert(["oat", "oath", "eat", "ear"])

insert('oat'  ) -> o:TẠO MỚI, a:TẠO MỚI, t:TẠO MỚI
insert('oath' ) -> o:dùng lại, a:dùng lại, t:dùng lại, h:TẠO MỚI
insert('eat'  ) -> e:TẠO MỚI, a:TẠO MỚI, t:TẠO MỚI
insert('ear'  ) -> e:dùng lại, a:dùng lại, r:TẠO MỚI
Tổng số node (không tính root): 8


### Insight nối Trie với lưới

Trong DFS trên lưới, thay vì mang theo **chuỗi** `s`, ta mang theo **node Trie** ứng với `s`:

| Cách 2 (chuỗi) | Trie (node) |
|---|---|
| `s = s + ch` — tạo chuỗi mới `O(L)` | `child = node.children.get(ch)` — `O(1)` |
| `s not in prefixes` — băm `O(L)` | `child is None` — `O(1)` |
| `s in word_set` — băm `O(L)` | `child.is_end` — `O(1)` |

Node Trie chính là "trạng thái tiền tố" được **nén** lại: đứng ở node `"oa"`, bước sang ô chữ `'t'` thì nhảy sang node `"oat"` ngay, không cần nhìn lại `"oa"`.

Thêm nữa: chỉ **một lần DFS** trên lưới phục vụ **mọi từ cùng lúc**, vì Trie chứa tất cả các từ.

## 9. Cách 3 (cách chính) — Trie + DFS một lần trên lưới

Thuật toán:

1. **Xây Trie** từ `words` (`insert` từng từ).
2. Với mỗi ô `(r, c)`, gọi `dfs(r, c, root)`.
3. `dfs(r, c, node)` — `node` ứng với chuỗi đã ghép **trước** ô `(r, c)`:
   - `ch = board[r][c]`; `child = node.children.get(ch)`. Nếu `child is None` (gồm cả trường hợp `ch == '#'`) → **cắt nhánh**, `return`.
   - Đẩy `ch` vào `path` (list ký tự). Nếu `child.is_end` → ghi nhận từ `"".join(path)` vào `set found`.
   - Đánh dấu `'#'`, gọi `dfs` sang 4 ô kề với `child`, trả lại chữ cũ, `path.pop()`.
4. Trả `list(found)`.

Vì sao dùng `path` là **list** + `append/pop` thay vì `s + ch`? Vì `append`/`pop` là `O(1)`; chỉ khi **tìm thấy từ** mới tốn `O(L)` để `join`.

Vì sao cần `set found`? Một từ có thể ghép được bằng nhiều đường (ví dụ lưới toàn `a`, từ `"aa"`), DFS sẽ "thấy" nó nhiều lần.

In [17]:
def build_trie(words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True
    return root

def find_words_trie_basic(board, words, stats=None):
    root = build_trie(words)
    m, n = len(board), len(board[0])
    found, path = set(), []

    def dfs(r, c, node):
        if stats is not None:
            stats["dfs"] += 1
        ch = board[r][c]
        child = node.children.get(ch)
        if child is None:                          # không từ nào đi tiếp được -> cắt nhánh
            return
        path.append(ch)
        if child.is_end:
            found.add("".join(path))
        board[r][c] = "#"                          # chọn
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                dfs(nr, nc, child)                 # đi tiếp (ô '#' sẽ tự bị cắt ở dòng get)
        board[r][c] = ch                           # bỏ chọn
        path.pop()

    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return list(found)

for b, w in [(board1, words1), (board2, words2)]:
    st = {"dfs": 0}
    print(sorted(find_words_trie_basic(copy.deepcopy(b), w, st)), "| bước DFS:", st["dfs"])

['eat', 'oath'] | bước DFS: 46
[] | bước DFS: 8


Cùng số bước DFS với cách 2 trên Example 1 — đúng như dự đoán: **cắt nhánh ở cùng chỗ** (đều dừng khi chuỗi không còn là tiền tố). Khác biệt nằm ở **chi phí mỗi bước** (`O(1)` thay vì `O(L)`) và **bộ nhớ** (Trie chia sẻ tiền tố).

### Còn chỗ nào lãng phí?

Xét lưới `4 x 4` toàn chữ `a` và `words = ["a", "aa", "aaa", ..., "aaaaaaaaaa"]` (10 từ). Ngay ở ô xuất phát đầu tiên, DFS đã tìm được **cả 10 từ** dọc một đường. Nhưng cách 3 **không biết** là mình đã xong: nó vẫn tiếp tục thử mọi đường đi khác từ cả 16 ô, liên tục "tìm thấy lại" những từ đã có (rồi `set` lặng lẽ bỏ trùng).

Ba điểm lãng phí:

1. Phải mang `path` và `join` mỗi lần thấy từ — chỉ để biết từ đó là gì.
2. Một từ bị tìm lại nhiều lần → cần `set`, và tốn bước.
3. Nhánh Trie của những từ **đã tìm hết** vẫn bị DFS đi vào.

## 10. Cách 4 — Tối ưu cách chính: lưu `word` tại node, xóa sau khi tìm, tỉa node lá

### Tối ưu (a): lưu cả từ tại node cuối

Thay cờ `is_end`, node cuối lưu luôn chuỗi: `node.word = "oath"` (node khác `word = None`). Tìm thấy thì có ngay từ, **không cần** `path`.

### Tối ưu (b): xóa `word` ngay khi tìm thấy

```python
if child.word is not None:
    result.append(child.word)
    child.word = None          # lần sau đi qua node này không ghi nhận lại
```

→ Mỗi từ được thêm **đúng một lần**, kết quả dùng `list` thường, **không cần `set`**.

### Tối ưu (c): tỉa node lá (pruning) khi quay lui

Sau khi DFS con trả về, nếu `child` **không còn con** và **không còn word** thì không còn gì để tìm dưới nó nữa → xóa khỏi cha:

```python
if not child.children and child.word is None:
    del node.children[ch]
```

Việc tỉa lan dần **từ dưới lên**: xóa lá `h` của `"oath"` → `t` trở thành lá rỗng → bị xóa khi quay về tầng trên → ... Lần sau DFS gặp chữ `'o'` ở ô khác, `root.children.get('o')` trả `None` → cắt nhánh **ngay bước đầu**.

Thêm một mẹo nhỏ: nếu `root.children` đã rỗng (tìm hết mọi từ) thì dừng toàn bộ vòng lặp.

In [18]:
class TrieNodeW:
    def __init__(self):
        self.children = {}
        self.word = None        # chuỗi đầy đủ nếu node là cuối một từ, ngược lại None

def build_trie_w(words):
    root = TrieNodeW()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNodeW())
        node.word = w                              # (a) lưu cả từ
    return root

def find_words_trie(board, words, stats=None):
    root = build_trie_w(words)
    m, n = len(board), len(board[0])
    result = []

    def dfs(r, c, node):
        if stats is not None:
            stats["dfs"] += 1
        ch = board[r][c]
        child = node.children.get(ch)
        if child is None:
            return
        if child.word is not None:
            result.append(child.word)
            child.word = None                      # (b) xóa để không tìm lại
        board[r][c] = "#"
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                dfs(nr, nc, child)
        board[r][c] = ch
        if not child.children and child.word is None:
            del node.children[ch]                  # (c) tỉa node lá đã "cạn"

    for r in range(m):
        for c in range(n):
            if not root.children:                  # đã tìm hết mọi từ
                return result
            dfs(r, c, root)
    return result

for b, w in [(board1, words1), (board2, words2)]:
    st = {"dfs": 0}
    print(find_words_trie(copy.deepcopy(b), w, st), "| bước DFS:", st["dfs"])

['oath', 'eat'] | bước DFS: 46
[] | bước DFS: 8


### Trie của Example 1 trước và sau khi chạy

Hàm `find_words_trie` tự xây Trie bên trong. Để quan sát, ta xây Trie bên ngoài rồi gọi phiên bản `find_words_on_trie(board, root)` (cùng logic, nhận sẵn `root`). Sau khi chạy, nhánh `oath` và `eat` bị tỉa sạch, chỉ còn `pea` và `rain` — hai từ **không** có trên lưới.

In [19]:
def find_words_on_trie(board, root):
    m, n = len(board), len(board[0])
    result = []

    def dfs(r, c, node):
        ch = board[r][c]
        child = node.children.get(ch)
        if child is None:
            return
        if child.word is not None:
            result.append(child.word)
            child.word = None
        board[r][c] = "#"
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                dfs(nr, nc, child)
        board[r][c] = ch
        if not child.children and child.word is None:
            del node.children[ch]

    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return result

root1 = build_trie_w(words1)
print("TRƯỚC:")
print_trie(root1)
print("\nKết quả:", find_words_on_trie(copy.deepcopy(board1), root1))
print("\nSAU (đã tỉa):")
print_trie(root1)

TRƯỚC:
(root)
├── o
│   └── a
│       └── t
│           └── h ★ "oath"
├── p
│   └── e
│       └── a ★ "pea"
├── e
│   └── a
│       └── t ★ "eat"
└── r
    └── a
        └── i
            └── n ★ "rain"

Kết quả: ['oath', 'eat']

SAU (đã tỉa):
(root)
├── p
│   └── e
│       └── a ★ "pea"
└── r
    └── a
        └── i
            └── n ★ "rain"


## 11. Trace cách 4 trên Example 1

Trie ban đầu có 4 nhánh từ root: `o` (oath), `p` (pea), `e` (eat), `r` (rain). Chỉ những ô có chữ thuộc `root.children` mới đi được quá bước đầu; các ô khác bị cắt ngay (trace không in chúng để gọn).

Bảng tóm tắt (khớp với output của cell bên dưới):

| Ô xuất phát | Đường đi (node Trie) | Sự kiện |
|---|---|---|
| `(0,0) o` | `"o" → "oa"(0,1) → "oat"(1,1) → "oath"(2,1)` | tìm thấy `"oath"`; quay lui tỉa `"oath"`, `"oat"`, `"oa"`, `"o"` (root mất nhánh `o`) |
| `(1,0) e` | `"e"` | ô kề là `o, i, t`, node `e` chỉ có con `a` → bế tắc |
| `(1,3) e` | `"e" → "ea"(1,2) → "eat"(1,1)` | tìm thấy `"eat"`; tỉa `"eat"`, `"ea"`, `"e"` |
| `(2,3) r` | `"r"` (cần `a`) | ô kề là `e, v, k`, không có `a` → bế tắc; nhánh `r` vẫn giữ (còn `rain`) |

Cột "node Trie" chính là tiền tố đang ghép. Nhánh `p` không bao giờ được chạm tới vì lưới không có chữ `p`.

In [20]:
def trace_find_words(board, words):
    root = build_trie_w(words)
    m, n = len(board), len(board[0])
    result = []
    print("root.children ban đầu:", list(root.children))

    def dfs(r, c, node, prefix, depth):
        ch = board[r][c]
        child = node.children.get(ch)
        if child is None:
            return False
        pad = "  " * depth
        msg = f"{pad}({r},{c}) '{ch}' -> node \"{prefix + ch}\""
        if child.word is not None:
            msg += f"   ★ TÌM THẤY \"{child.word}\""
            result.append(child.word)
            child.word = None
        print(msg)
        board[r][c] = "#"
        moved = False
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                moved = dfs(nr, nc, child, prefix + ch, depth + 1) or moved
        board[r][c] = ch
        if not moved and child.children:
            print(f"{pad}  (bế tắc: không ô kề nào khớp {list(child.children)})")
        if not child.children and child.word is None:
            del node.children[ch]
            print(f"{pad}  tỉa node \"{prefix + ch}\"")
        return True

    for r in range(m):
        for c in range(n):
            if board[r][c] in root.children:
                print(f"-- xuất phát ({r},{c})")
                dfs(r, c, root, "", 1)
    print("root.children cuối cùng:", list(root.children))
    return result

print("Kết quả:", trace_find_words(copy.deepcopy(board1), words1))

root.children ban đầu: ['o', 'p', 'e', 'r']
-- xuất phát (0,0)
  (0,0) 'o' -> node "o"
    (0,1) 'a' -> node "oa"
      (1,1) 't' -> node "oat"
        (2,1) 'h' -> node "oath"   ★ TÌM THẤY "oath"
          tỉa node "oath"
        tỉa node "oat"
      tỉa node "oa"
    tỉa node "o"
-- xuất phát (1,0)
  (1,0) 'e' -> node "e"
    (bế tắc: không ô kề nào khớp ['a'])
-- xuất phát (1,3)
  (1,3) 'e' -> node "e"
    (1,2) 'a' -> node "ea"
      (1,1) 't' -> node "eat"   ★ TÌM THẤY "eat"
        tỉa node "eat"
      tỉa node "ea"
    tỉa node "e"
-- xuất phát (2,3)
  (2,3) 'r' -> node "r"
    (bế tắc: không ô kề nào khớp ['a'])
root.children cuối cùng: ['p', 'r']
Kết quả: ['oath', 'eat']


Điểm đáng chú ý trong trace:

- Khi tìm thấy `"oath"`, node `h` không còn con và `word` đã về `None` → bị tỉa. Quay về `t`: giờ `t` cũng rỗng → tỉa. Cứ thế lên tới `o`. Nhờ vậy `root.children` cuối cùng không còn `o` và `e`.
- Ô `(1,3) e` tìm `"eat"` đi qua đúng ô `t(1,1)` mà `"oath"` đã dùng — hợp lệ, vì đây là **đường đi khác** (dấu `'#'` của lần trước đã được trả lại).
- Nhánh `r` không bị tỉa vì node `r` vẫn còn con `a` (từ `"rain"` chưa tìm thấy). Tỉa chỉ xảy ra khi **chắc chắn** không còn gì để tìm.

## 12. So sánh số bước DFS trước/sau tối ưu

Ta đo tổng số lần gọi `dfs` của 4 cách trên vài bộ input (số bước là **tất định**, chạy lại vẫn ra đúng con số đó):

1. `Example 1`.
2. `all-a 4x4`: lưới 4x4 toàn `a`, words `a, aa, ..., a×10` — mọi từ đều có.
3. `all-a 4x4 + bẫy`: như trên nhưng từ dài nhất là `a×9 + b` (không có trên lưới) — một nhánh Trie dài **không bao giờ** được tìm thấy nên không thể tỉa.
4. `ngẫu nhiên 8x8`: chữ trong `abcd`, khoảng 300 từ ngẫu nhiên dài 3–7 (`random.seed(212)`).

In [21]:
import random

def count_steps(f, board, words):
    st = {"dfs": 0}
    res = f(copy.deepcopy(board), words, st)
    return st["dfs"], len(res)

all_a = [["a"] * 4 for _ in range(4)]
random.seed(212)
rand_board = [[random.choice("abcd") for _ in range(8)] for _ in range(8)]
rand_words = sorted({"".join(random.choice("abcd") for _ in range(random.randint(3, 7))) for _ in range(300)})

tests = [
    ("Example 1",          board1, words1),
    ("all-a 4x4",          all_a,  ["a" * k for k in range(1, 11)]),
    ("all-a 4x4 + bẫy",    all_a,  ["a" * k for k in range(1, 10)] + ["a" * 9 + "b"]),
    ("ngẫu nhiên 8x8",     rand_board, rand_words),
]
methods = [("C1 từng từ", word_search_each), ("C2 prefix set", find_words_prefix_set),
           ("C3 Trie cơ bản", find_words_trie_basic), ("C4 Trie tối ưu", find_words_trie)]

print(f"{'input':18}" + "".join(f"{name:>16}" for name, _ in methods) + "   số từ tìm được")
for tname, b, w in tests:
    counts = [count_steps(f, b, w) for _, f in methods]
    assert len({k for _, k in counts}) == 1          # mọi cách tìm cùng số từ
    print(f"{tname:18}" + "".join(f"{s:>16}" for s, _ in counts) + f"   {counts[0][1]}")

input                   C1 từng từ   C2 prefix set  C3 Trie cơ bản  C4 Trie tối ưu   số từ tìm được
Example 1                       58              46              46              46   2
all-a 4x4                       80           33160           33160              32   10
all-a 4x4 + bẫy              22361           22296           22296           22296   9


ngẫu nhiên 8x8               30187           10959           10959            6400   194


Đọc bảng:

- **C2 và C3 luôn bằng nhau** về số bước: cả hai cắt nhánh ở đúng cùng chỗ. Trie thắng ở **chi phí mỗi bước** và **bộ nhớ**, không phải số bước.
- **`all-a 4x4`**: C4 chỉ cần **32** bước so với **33160** của C3. Dọc đường đi đầu tiên C4 tìm đủ 10 từ, tỉa sạch Trie, `root.children` rỗng → dừng toàn bộ. C3 thì tiếp tục đi mọi đường và "tìm lại" các từ vô số lần.
- **`all-a 4x4 + bẫy`**: C4 **không** đỡ hơn C3 (cùng 22296), vì nhánh `a...ab` không bao giờ được tìm thấy nên không thể tỉa. Đây là kiểu trường hợp xấu nhất: tối ưu (c) không đổi Big-O worst case, nó chỉ giúp rất nhiều trong thực tế.
- **Ngẫu nhiên 8x8**: C4 giảm khoảng 40% số bước so với C2/C3 (6400 so với 10959), và ít hơn C1 gần 5 lần (30187).
- C1 có lúc thắng C2/C3 (`all-a 4x4`: chỉ 80 bước, vì `exist` dừng ngay khi thấy từ — một kiểu "dừng sớm" mà C3 không có, còn C4 thì có), nhưng khi nhiều từ thì C1 tệ nhất vì mỗi từ một lần quét lưới — với `W = 3·10^4` từ thì không thể dùng.

## 13. Code nộp LeetCode — cách chính (Trie + DFS + 3 tối ưu)

Chuyển `find_words_trie` vào `class Solution`:

- `TrieNode` để ngoài class (LeetCode cho phép định nghĩa thêm class).
- `dfs` lồng trong `findWords` để dùng chung `board`, `m`, `n`, `result`.
- Độ sâu đệ quy tối đa `L + 1 <= 11` → không lo tràn call stack của Python.

In [22]:
class TrieNode212:
    __slots__ = ("children", "word")           # tiết kiệm bộ nhớ khi có tới 3*10^5 node
    def __init__(self):
        self.children = {}
        self.word = None

class Solution:
    def findWords(self, board: List[List[str]], words: List[str]) -> List[str]:
        root = TrieNode212()
        for w in words:
            node = root
            for ch in w:
                node = node.children.setdefault(ch, TrieNode212())
            node.word = w

        m, n = len(board), len(board[0])
        result = []

        def dfs(r, c, parent):
            ch = board[r][c]
            node = parent.children.get(ch)
            if node is None:                       # sai chữ hoặc ô '#'
                return
            if node.word is not None:
                result.append(node.word)
                node.word = None                   # không tìm lại
            board[r][c] = "#"
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    dfs(nr, nc, node)
            board[r][c] = ch
            if not node.children and node.word is None:
                del parent.children[ch]            # tỉa lá

        for r in range(m):
            for c in range(n):
                if not root.children:
                    return result
                dfs(r, c, root)
        return result

### Bộ test

Ngoài 2 ví dụ đề, ta thử các trường hợp dễ sai:

| Ca | Vì sao đáng thử |
|---|---|
| lưới `1x1` | biên nhỏ nhất; từ dài hơn số ô |
| `"oat"` và `"oath"` cùng lúc | từ này là tiền tố của từ kia — không được dừng sớm |
| lưới `[["a","a"]]`, từ `"aaa"` | cần 3 ô nhưng chỉ có 2 → không dùng lại ô |
| nhiều đường cho cùng một từ | không được trả trùng |
| tất cả từ đều có | kiểm tra dừng sớm khi Trie rỗng |
| lưới không bị thay đổi sau khi chạy | mọi `'#'` đều được trả lại |

In [23]:
def check(name, board, words, expected):
    b = copy.deepcopy(board)
    got = Solution().findWords(b, words)
    assert sorted(got) == sorted(expected), (name, got, expected)
    assert len(got) == len(set(got)), (name, "bị trùng", got)
    assert b == board, (name, "lưới bị thay đổi!")
    print(f"{name:28} PASS -> {sorted(got)}")

check("Example 1", board1, words1, ["eat", "oath"])
check("Example 2", board2, words2, [])
check("1x1 có", [["a"]], ["a", "b"], ["a"])
check("1x1 từ quá dài", [["a"]], ["aa"], [])
check("tiền tố oat + oath", board1, ["oat", "oath", "oaths"], ["oat", "oath"])
check("không dùng lại ô", [["a", "a"]], ["aa", "aaa"], ["aa"])
check("nhiều đường cùng một từ", [["a", "b"], ["b", "a"]], ["ab", "ba", "aba"], ["ab", "ba", "aba"])
check("tìm hết, dừng sớm", [["a", "b"], ["c", "d"]], ["ab", "abd", "acdb", "dcab", "ba"],
      ["ab", "abd", "acdb", "dcab", "ba"])
check("lưới 12x12 toàn 'a'", [["a"] * 12 for _ in range(12)], ["a" * 10, "a" * 9 + "b"], ["a" * 10])

Example 1                    PASS -> ['eat', 'oath']
Example 2                    PASS -> []
1x1 có                       PASS -> ['a']
1x1 từ quá dài               PASS -> []
tiền tố oat + oath           PASS -> ['oat', 'oath']
không dùng lại ô             PASS -> ['aa']
nhiều đường cùng một từ      PASS -> ['ab', 'aba', 'ba']
tìm hết, dừng sớm            PASS -> ['ab', 'abd', 'acdb', 'ba', 'dcab']


lưới 12x12 toàn 'a'          PASS -> ['aaaaaaaaaa']


### Đối chiếu chéo các cách (input ngẫu nhiên, `random.seed` cố định)

Sinh 300 bộ input nhỏ (lưới tới `4x4`, bảng chữ nhỏ `ab` hoặc `abc` để các từ hay xuất hiện), chạy **cả 5 hàm**: brute force sinh mọi chuỗi (`find_words_all_paths`), cách 1, cách 2, cách 3, cách 4 và `Solution`. Tất cả phải cho cùng tập kết quả.

In [24]:
random.seed(2024)
all_methods = [find_words_all_paths, word_search_each, find_words_prefix_set,
               find_words_trie_basic, find_words_trie,
               lambda b, w, stats=None: Solution().findWords(b, w)]
found_total = 0
for trial in range(300):
    m, n = random.randint(1, 4), random.randint(1, 4)
    alphabet = random.choice(["ab", "abc"])
    b = [[random.choice(alphabet) for _ in range(n)] for _ in range(m)]
    w = list({"".join(random.choice(alphabet) for _ in range(random.randint(1, 6)))
              for _ in range(random.randint(1, 12))})
    answers = [sorted(f(copy.deepcopy(b), w)) for f in all_methods]
    assert all(a == answers[0] for a in answers), (b, w, answers)
    found_total += len(answers[0])
print("300 bộ ngẫu nhiên: 6 cài đặt khớp nhau. Tổng số từ tìm được:", found_total)

300 bộ ngẫu nhiên: 6 cài đặt khớp nhau. Tổng số từ tìm được: 708


## 14. Vì sao cách chính đúng? (invariant)

**Invariant (bất biến) của `dfs(r, c, parent)`**: tại thời điểm gọi,

1. `parent` là node Trie ứng với đúng chuỗi ghép từ các ô trên đường đi hiện tại (chưa tính `(r, c)`);
2. mọi ô trên đường đi hiện tại đang mang dấu `'#'`, mọi ô khác giữ chữ gốc.

Từ đó:

- **Không trả nhầm**: một từ chỉ được thêm khi đi tới node có `word`, tức chuỗi trên đường đi **bằng** từ đó; đường đi gồm các ô kề nhau (ta chỉ bước sang 4 ô kề) và **không lặp ô** (ô đã dùng là `'#'`, không có trong `children` nào).
- **Không bỏ sót**: giả sử từ `w` ghép được bằng đường `p1, p2, ..., pk`. Khi vòng lặp ngoài tới `p1`, mỗi tiền tố `w[:i]` đều có node trong Trie (vì `w` đã được `insert`), nên DFS không bị cắt và đi được hết `p1 → ... → pk`. Ngoại lệ duy nhất: node bị tỉa hoặc `word` bị xóa — điều đó chỉ xảy ra khi `w` **đã được ghi nhận** trước đó.
- **Tỉa an toàn**: ta chỉ xóa node khi nó **không còn con và không còn word**. Một node như vậy không thể dẫn tới từ nào chưa tìm, nên xóa nó không làm mất kết quả.
- **Không trùng**: `word` bị đặt `None` ngay lần đầu tìm thấy, nên mỗi từ được `append` tối đa một lần.
- **Lưới nguyên vẹn**: mỗi `board[r][c] = "#"` có đúng một `board[r][c] = ch` tương ứng trước khi hàm trả về.

## 15. Độ phức tạp của cách chính

Nhắc lại ký hiệu: lưới `m x n`, `W` từ, độ dài tối đa `L`, tổng số ký tự `S`.

**Thời gian**

- Xây Trie: mỗi ký tự của mỗi từ đi qua một node → `O(S)`.
- DFS: từ mỗi ô, đường đi dài tối đa `L` (vì Trie sâu tối đa `L`), bước đầu 4 hướng, các bước sau tối đa 3 hướng → `O(m·n·4·3^(L-1))` trong trường hợp xấu nhất. Mỗi bước `O(1)` (một lần tra `dict`).
- Tổng: `O(S + m·n·4·3^(L-1))`. Với constraints: `S <= 3·10^5`, `m·n·4·3^9 <= 144·4·19683 ≈ 1,1·10^7`.

So với cách 1 `O(W·m·n·4·3^(L-1))`: ta bỏ hẳn được thừa số `W` khỏi phần DFS — đó là ý nghĩa của "DFS **một lần** cho mọi từ".

Thực tế còn nhanh hơn nhiều cận trên, vì Trie cắt nhánh ngay khi tiền tố không tồn tại và tối ưu (c) xóa dần những nhánh đã xong.

**Bộ nhớ**

- Trie: tối đa `S` node (thường ít hơn nhờ chia sẻ tiền tố) → `O(S)`.
- Call stack: độ sâu tối đa `L + 1` → `O(L)`.
- Kết quả: `O(S)` trong trường hợp mọi từ đều có.
- Không dùng thêm `visited` vì ghi đè `'#'` ngay trên lưới.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian (worst) | Bộ nhớ thêm | Ưu / nhược |
|---|---|---|---|---|
| 1. Word Search I cho từng từ | `exist(board, w)` với mọi `w` | `O(W·m·n·4·3^(L-1))` | `O(L)` | dễ viết nhất; lặp việc cho từ chung tiền tố; TLE với `W` lớn |
| 2a. Sinh mọi chuỗi `<= L` | DFS từ mỗi ô, tra `word_set` | `O(m·n·4·3^(L-1)·L)` | `O(S)` | không cắt nhánh, đi cả đường vô vọng |
| 2b. + tập tiền tố | dừng khi `s not in prefixes` | `O(m·n·4·3^(L-1)·L)` | `O(S·L)` | cắt nhánh tốt; mỗi bước tạo/băm chuỗi `O(L)` |
| 3. Trie + DFS (cách chính) | mang node Trie thay cho chuỗi | `O(S + m·n·4·3^(L-1))` | `O(S)` | mỗi bước `O(1)`; vẫn tìm lại từ cũ, cần `set` |
| 4. Trie + 3 tối ưu | `word` tại node, xóa khi thấy, tỉa lá | như cách 3 (worst) | `O(S)` | nhanh nhất thực tế; bản nên nộp |

Lộ trình tư duy: **mỗi từ một lần quét** (1) → **một lần quét cho mọi từ, tra bằng chuỗi** (2) → **thay chuỗi bằng con trỏ Trie** (3) → **dọn dẹp những gì đã xong** (4).

## 17. Lỗi thường gặp

### Lỗi 1: Quên trả lại chữ cũ sau khi đánh dấu `'#'`

```python
board[r][c] = "#"
for ...:
    if dfs(...): return True
return False            # ✘ quên board[r][c] = ch
```

Lần xuất phát đầu thất bại để lại "lỗ `#`" trên lưới, làm các lần xuất phát sau bị chặn oan. Ví dụ lưới `[["a","b"],["a","c"]]`, từ `"aab"` có đường `(1,0) a → (0,0) a → (0,1) b`. Bản lỗi: xuất phát `(0,0)` đi sang `(1,0)` rồi bế tắc, cả hai ô bị bỏ lại là `'#'` → lần xuất phát `(1,0)` thất bại.

In [25]:
def exist_bug_no_restore(board, word):
    m, n = len(board), len(board[0])
    def dfs(r, c, i):
        if board[r][c] != word[i]:
            return False
        if i == len(word) - 1:
            return True
        board[r][c] = "#"                         # đánh dấu nhưng KHÔNG BAO GIỜ trả lại
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and dfs(nr, nc, i + 1):
                return True
        return False
    return any(dfs(r, c, 0) for r in range(m) for c in range(n))

g = [["a", "b"], ["a", "c"]]
b = copy.deepcopy(g)
print("Bản lỗi :", exist_bug_no_restore(b, "aab"), "| lưới sau:", b)
b = copy.deepcopy(g)
print("Bản đúng:", exist(b, "aab"), " | lưới sau:", b)

Bản lỗi : False | lưới sau: [['#', 'b'], ['#', 'c']]
Bản đúng: True  | lưới sau: [['a', 'b'], ['a', 'c']]


**Sửa**: mọi lối ra khỏi hàm sau khi đã đánh dấu đều phải trả lại `board[r][c] = ch` (kể cả nhánh `return True`).

### Lỗi 2: Không đánh dấu ô đang dùng

Nếu bỏ hẳn bước `'#'`, DFS có thể đi `b → a → b` quay lại ô cũ. Example 2 `"abcb"` thì không đủ để lộ lỗi (vì `b` không kề `c`), nên ta lấy ví dụ `"aba"` trên lưới `[["a","b"]]`: chỉ có 2 ô nhưng bản lỗi "ghép" được 3 chữ.

In [26]:
def exist_bug_no_mark(board, word):
    m, n = len(board), len(board[0])
    def dfs(r, c, i):
        if board[r][c] != word[i]:
            return False
        if i == len(word) - 1:
            return True
        return any(dfs(r + dr, c + dc, i + 1) for dr, dc in DIRS
                   if 0 <= r + dr < m and 0 <= c + dc < n)      # không có '#'
    return any(dfs(r, c, 0) for r in range(m) for c in range(n))

print("Bản lỗi  'aba' trên [['a','b']]:", exist_bug_no_mark([["a", "b"]], "aba"))
print("Bản đúng 'aba' trên [['a','b']]:", exist([["a", "b"]], "aba"))

Bản lỗi  'aba' trên [['a','b']]: True
Bản đúng 'aba' trên [['a','b']]: False


**Sửa**: đánh dấu ô **trước** khi đi sang ô kề (`board[r][c] = "#"`), trả lại khi quay về.

### Lỗi 3: Dùng `list` + `is_end` mà không chống trùng

Lưới `[["a","b"],["b","a"]]`, từ `"ab"`: có 4 đường (`a(0,0)→b(0,1)`, `a(0,0)→b(1,0)`, `a(1,1)→b(0,1)`, `a(1,1)→b(1,0)`). Bản lỗi trả `"ab"` 4 lần — LeetCode báo **Wrong Answer**.

In [27]:
def find_words_bug_duplicates(board, words):
    root = build_trie(words)
    m, n = len(board), len(board[0])
    result, path = [], []
    def dfs(r, c, node):
        ch = board[r][c]
        child = node.children.get(ch)
        if child is None:
            return
        path.append(ch)
        if child.is_end:
            result.append("".join(path))          # ✘ list, không xóa, không set
        board[r][c] = "#"
        for nr, nc in neighbors(board, r, c):
            dfs(nr, nc, child)
        board[r][c] = ch
        path.pop()
    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return result

g = [["a", "b"], ["b", "a"]]
print("Bản lỗi :", find_words_bug_duplicates(copy.deepcopy(g), ["ab"]))
print("Bản đúng:", find_words_trie(copy.deepcopy(g), ["ab"]))

Bản lỗi : ['ab', 'ab', 'ab', 'ab']
Bản đúng: ['ab']


**Sửa**: hoặc dùng `set` (cách 3), hoặc đặt `child.word = None` ngay khi tìm thấy (cách 4).

### Lỗi 4: `return` ngay khi tìm thấy một từ

Người mới quen với #79 ("thấy là xong") nên viết `if child.word: result.append(...); return`. Nhưng `"oat"` là tiền tố của `"oath"`: dừng ở `t` thì không bao giờ đi tới `h`.

In [28]:
def find_words_bug_early_return(board, words):
    root = build_trie_w(words)
    m, n = len(board), len(board[0])
    result = []
    def dfs(r, c, node):
        ch = board[r][c]
        child = node.children.get(ch)
        if child is None:
            return
        if child.word is not None:
            result.append(child.word)
            child.word = None
            return                                 # ✘ dừng sớm: bỏ lỡ từ dài hơn
        board[r][c] = "#"
        for nr, nc in neighbors(board, r, c):
            dfs(nr, nc, child)
        board[r][c] = ch
    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return result

print("Bản lỗi :", sorted(find_words_bug_early_return(copy.deepcopy(board1), ["oat", "oath"])))
print("Bản đúng:", sorted(find_words_trie(copy.deepcopy(board1), ["oat", "oath"])))

Bản lỗi : ['oat']
Bản đúng: ['oat', 'oath']


**Sửa**: ghi nhận từ rồi **vẫn đi tiếp** xuống các ô kề; chỉ dừng khi `child is None`.

### Lỗi 5: Tỉa sai điều kiện — xóa node khi nó vẫn còn con

Chỉ kiểm tra `child.word is None` mà quên `not child.children`. Lưới một hàng `b a x a b c`, words `["ab", "abc"]`:

- Xuất phát `a(0,1)`: sang `b(0,0)` → tìm thấy `"ab"`, nhưng `b` không kề `c` nào.
- Bản lỗi: quay về thấy node `b` có `word None` → xóa luôn, **dù node `b` còn con `c`** (của `"abc"`).
- Xuất phát `a(0,3)`: đường `a → b(0,4) → c(0,5)` lẽ ra cho `"abc"`, nhưng nhánh `b` đã mất.

In [29]:
def find_words_bug_prune(board, words):
    root = build_trie_w(words)
    m, n = len(board), len(board[0])
    result = []
    def dfs(r, c, node):
        ch = board[r][c]
        child = node.children.get(ch)
        if child is None:
            return
        if child.word is not None:
            result.append(child.word)
            child.word = None
        board[r][c] = "#"
        for nr, nc in neighbors(board, r, c):
            dfs(nr, nc, child)
        board[r][c] = ch
        if child.word is None:                     # ✘ thiếu: and not child.children
            del node.children[ch]
    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return result

row = [["b", "a", "x", "a", "b", "c"]]
print("Bản lỗi :", sorted(find_words_bug_prune(copy.deepcopy(row), ["ab", "abc"])))
print("Bản đúng:", sorted(find_words_trie(copy.deepcopy(row), ["ab", "abc"])))

Bản lỗi : ['ab']
Bản đúng: ['ab', 'abc']


**Sửa**: chỉ tỉa khi **cả hai** điều kiện đúng: `not child.children and child.word is None`.

### Lỗi 6 (hiệu năng): copy lưới hoặc `visited` mới cho mỗi bước

Viết `dfs(nr, nc, child, visited | {(r, c)})` hoặc `copy.deepcopy(board)` trong mỗi lời gọi tạo set/lưới mới ở **mỗi bước** → chậm hơn nhiều lần và dễ TLE. Dùng một cấu trúc chung + đánh dấu/bỏ đánh dấu (backtracking) là đủ.

Tương tự, nhiều bạn kiểm tra `len(path) > 10` hay giới hạn độ dài thủ công — không cần, vì Trie tự cắt khi hết chiều sâu.

## 18. Mẹo nhận dạng pattern

| Dấu hiệu trong đề | Nghĩ tới |
|---|---|
| Lưới chữ, ghép từ bằng ô **kề nhau**, không dùng lại ô | DFS + backtracking, đánh dấu `'#'` rồi trả lại |
| Chỉ **một** từ cần tìm | #79 Word Search — DFS theo chỉ số `i` của từ |
| **Rất nhiều** từ, nhiều từ chung phần đầu | **Trie** để tìm tất cả trong một lần duyệt |
| Câu hỏi "có từ nào **bắt đầu bằng** ...?" | Trie / `startsWith` |
| Kết quả "tìm tất cả, không trùng" | lưu `word` tại node + xóa sau khi thấy |
| Thấy TLE dù đã dùng Trie | tỉa node lá, dừng khi `root.children` rỗng |

Câu hỏi tự đặt: *"Mình có đang lặp lại cùng một việc cho nhiều chuỗi có chung tiền tố không?"* Nếu có → Trie.

## 19. Biến thể & bài liên quan

- **#79 Word Search** (Medium): bài nền — tìm một từ, trả `True/False`.
- **#208 Implement Trie (Prefix Tree)** (Medium): đúng 3 hàm `insert / search / startsWith` ở mục 8.
- **#211 Design Add and Search Words Data Structure** (Medium): Trie + tìm với ký tự đại diện `'.'` (DFS trên Trie) — xem Bài tập 5.
- **#1268 Search Suggestions System** (Medium): Trie gợi ý từ theo tiền tố gõ dần.
- **#648 Replace Words** (Medium): thay từ bằng gốc ngắn nhất — đi Trie tới node `word` đầu tiên.
- **#472 Concatenated Words** (Hard): Trie/DP kiểm tra một từ ghép từ các từ khác.
- **#1032 Stream of Characters** (Hard): Trie của các từ **đảo ngược**.
- **#421 Maximum XOR of Two Numbers in an Array** (Medium): Trie trên **bit** thay vì chữ.
- **#200 Number of Islands** (Medium): DFS trên lưới với visited vĩnh viễn (như flood fill ở mục 3).

## 20. Bài tập vận dụng

Làm theo thứ tự, tự giải trước rồi mới xem **Gợi ý** (mục 21) và **Đáp án** (mục 22).

### Bài 1 — Tính tay (không chạy code)

Vẽ Trie của `["cat", "car", "cart", "dog", "do"]`.

a) Trie có bao nhiêu node (không tính root)?
b) Bao nhiêu node được đánh dấu `★`?
c) Node `r` (của `"car"`) có phải node lá không?

### Bài 2 — Đoán output

Với lưới `[["a","b"],["c","d"]]` và `words = ["ab", "abd", "acdb", "abcd", "ba", "dcab"]`, `sorted(Solution().findWords(...))` in ra gì? Giải thích vì sao từ bị loại không ghép được.

### Bài 3 — Điền TODO: cài đặt Trie (#208)

Hoàn thiện class `ExerciseTrie` với `insert`, `search`, `startsWith`.

In [30]:
class ExerciseNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class ExerciseTrie:
    def __init__(self):
        self.root = ExerciseNode()

    def insert(self, word: str) -> None:
        # TODO 1: node = root; với mỗi ch, nếu chưa có con thì tạo; đi xuống
        # TODO 2: đánh dấu is_end ở node cuối
        pass

    def search(self, word: str) -> bool:
        # TODO 3: đi theo từng ký tự, gãy -> False; hết -> node.is_end
        raise NotImplementedError

    def startsWith(self, prefix: str) -> bool:
        # TODO 4: giống search nhưng đi hết là True
        raise NotImplementedError

# Tự làm xong rồi mới mở test:
# tr = ExerciseTrie()
# tr.insert("apple")
# assert tr.search("apple") is True
# assert tr.search("app") is False
# assert tr.startsWith("app") is True
# tr.insert("app")
# assert tr.search("app") is True

### Bài 4 — Tự viết lại cách chính

Viết `exercise_find_words(board, words)` theo cách 4 (Trie lưu `word`, xóa khi tìm thấy, tỉa lá). Cố gắng **không nhìn** mục 10/13.

In [31]:
def exercise_find_words(board, words):
    # TODO 1: xây Trie, node cuối lưu word
    # TODO 2: viết dfs(r, c, parent): get child; None -> return
    # TODO 3: thấy word -> append, đặt None; đánh dấu '#', đi 4 hướng, trả lại chữ
    # TODO 4: tỉa: nếu child không còn con và không còn word -> xóa khỏi parent
    # TODO 5: gọi dfs từ mọi ô
    pass

# Tự làm xong rồi mới mở test:
# assert sorted(exercise_find_words(copy.deepcopy(board1), words1)) == ["eat", "oath"]
# assert exercise_find_words(copy.deepcopy(board2), words2) == []
# assert sorted(exercise_find_words([["a", "b"], ["b", "a"]], ["ab", "aba"])) == ["ab", "aba"]

### Bài 5 — Biến thể: #211 tìm kiếm với ký tự đại diện `'.'`

Cài `WordDictionary` với `addWord(word)` và `search(word)`, trong đó `'.'` khớp **bất kỳ một** chữ cái. Ví dụ đề #211:

```text
["WordDictionary","addWord","addWord","addWord","search","search","search","search"]
[[],["bad"],["dad"],["mad"],["pad"],["bad"],[".ad"],["b.."]]
Output: [null,null,null,null,false,true,true,true]
```

Gợi ý nhanh: đây là DFS — nhưng trên **Trie** thay vì trên lưới.

In [32]:
class WordDictionary:
    def __init__(self):
        # TODO: tạo root
        pass

    def addWord(self, word: str) -> None:
        # TODO: giống insert
        pass

    def search(self, word: str) -> bool:
        # TODO: dfs(i, node); gặp '.' thì thử mọi node con
        raise NotImplementedError

# Tự làm xong rồi mới mở test:
# wd = WordDictionary()
# for w in ["bad", "dad", "mad"]:
#     wd.addWord(w)
# assert [wd.search(q) for q in ["pad", "bad", ".ad", "b.."]] == [False, True, True, True]

### Bài 6 — Mini challenge: trả về cả đường đi

Viết `find_words_with_paths(board, words)` trả về `dict`: từ → **một** đường đi (list tọa độ) ghép ra từ đó. Ví dụ với Example 1, `"eat"` → `[(1, 3), (1, 2), (1, 1)]`. Hàm kiểm tra `is_valid_path` cho sẵn bên dưới (chạy được ngay).

In [33]:
def is_valid_path(board, word, path):
    if len(path) != len(word) or len(set(path)) != len(path):
        return False                                   # sai độ dài hoặc lặp ô
    for k, (r, c) in enumerate(path):
        if board[r][c] != word[k]:
            return False
        if k > 0 and abs(r - path[k - 1][0]) + abs(c - path[k - 1][1]) != 1:
            return False                               # không kề nhau
    return True

print(is_valid_path(board1, "eat", [(1, 3), (1, 2), (1, 1)]))    # True
print(is_valid_path(board1, "eat", [(1, 3), (1, 2), (2, 2)]))    # False: (2,2) là 'k'

def find_words_with_paths(board, words):
    # TODO: như cách 4 nhưng mang theo list tọa độ path (append/pop)
    # TODO: khi thấy word -> lưu result[word] = path[:]  (nhớ COPY!)
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# res = find_words_with_paths(copy.deepcopy(board1), words1)
# assert sorted(res) == ["eat", "oath"]
# assert all(is_valid_path(board1, w, p) for w, p in res.items())

True
False


## 21. Gợi ý theo mức

**Bài 1**
- Mức 1: chèn từng từ, đếm node **mới** được tạo ở mỗi lần chèn.
- Mức 2: `"cart"` dùng lại `c → a → r` của `"car"`; `"do"` dùng lại `d → o` của `"dog"`.
- Mức 3: số node mới lần lượt là 3, 1, 1, 3, 0.

**Bài 2**
- Mức 1: vẽ lưới, các ô kề của `b(0,1)` là `a(0,0)` và `d(1,1)`.
- Mức 2: `"abcd"` cần đi `b → c`, nhưng `b(0,1)` và `c(1,0)` chéo nhau.
- Mức 3: chỉ có đúng 1 từ bị loại.

**Bài 3**
- Mức 1: cả 3 hàm đều "đi từ root theo từng ký tự".
- Mức 2: `node = node.children.get(ch)`; nếu `None` thì gãy.
- Mức 3: `search` trả `node.is_end`, `startsWith` trả `True` khi đi hết.

**Bài 4**
- Mức 1: chép lại khuôn "chọn → đi tiếp → bỏ chọn", thêm bước tỉa **sau** khi bỏ chọn.
- Mức 2: `dfs` nhận `parent`, vì cần `del parent.children[ch]`.
- Mức 3: điều kiện tỉa là `not child.children and child.word is None`.

**Bài 5**
- Mức 1: chữ thường → đi đúng một con; `'.'` → thử **mọi** con, chỉ cần một nhánh `True`.
- Mức 2: viết `dfs(i, node)`: `i == len(word)` thì trả `node.is_end`.
- Mức 3: `any(dfs(i + 1, child) for child in node.children.values())` cho trường hợp `'.'`.

**Bài 6**
- Mức 1: `path` là list tọa độ, `append((r, c))` khi vào, `pop()` khi ra — đặt cạnh việc đánh dấu `'#'`.
- Mức 2: lưu `path[:]` chứ không lưu `path` (list sẽ bị `pop` dần về rỗng!).
- Mức 3: vì đặt `word = None` sau lần đầu, mỗi từ chỉ có một đường được lưu.

## 22. Đáp án tham khảo

### Đáp án Bài 1

```text
(root)
├── c
│   └── a
│       ├── t ★ "cat"
│       └── r ★ "car"
│           └── t ★ "cart"
└── d
    └── o ★ "do"
        └── g ★ "dog"
```

a) **8** node; b) **5** node `★`; c) **Không** — node `r` là từ `"car"` nhưng còn con `t`. Kiểm tra bằng code:

In [34]:
def count_nodes(node):
    return sum(1 + count_nodes(ch) for ch in node.children.values())

def count_words(node):
    return sum(int(ch.is_end) + count_words(ch) for ch in node.children.values())

t1 = Trie()
for w in ["cat", "car", "cart", "dog", "do"]:
    t1.insert(w)
print_trie(t1.root)
r_node = t1._walk("car")
print("Số node:", count_nodes(t1.root), "| số ★:", count_words(t1.root),
      "| node 'car' là lá?", not r_node.children)
assert (count_nodes(t1.root), count_words(t1.root)) == (8, 5)

(root)
├── c
│   └── a
│       ├── t ★ "cat"
│       └── r ★ "car"
│           └── t ★ "cart"
└── d
    └── o ★ "do"
        └── g ★ "dog"
Số node: 8 | số ★: 5 | node 'car' là lá? False


### Đáp án Bài 2

`["ab", "abd", "acdb", "ba", "dcab"]`.

- `"ab"`: `a(0,0) → b(0,1)`; `"abd"`: thêm `d(1,1)`; `"acdb"`: `a → c(1,0) → d(1,1) → b(0,1)`; `"ba"`: `b(0,1) → a(0,0)`; `"dcab"`: `d(1,1) → c(1,0) → a(0,0) → b(0,1)`.
- `"abcd"` bị loại: lưới chỉ có một `a` và một `b`, mà `b(0,1)` không kề `c(1,0)`.

In [35]:
ans2 = sorted(Solution().findWords([["a", "b"], ["c", "d"]], ["ab", "abd", "acdb", "abcd", "ba", "dcab"]))
print(ans2)
assert ans2 == ["ab", "abd", "acdb", "ba", "dcab"]

['ab', 'abd', 'acdb', 'ba', 'dcab']


In [36]:
# Đáp án Bài 3
class AnswerTrie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word: str) -> bool:
        node = self._walk(word)
        return node is not None and node.is_end

    def startsWith(self, prefix: str) -> bool:
        return self._walk(prefix) is not None

tr = AnswerTrie()
tr.insert("apple")
assert tr.search("apple") is True
assert tr.search("app") is False
assert tr.startsWith("app") is True
tr.insert("app")
assert tr.search("app") is True
print("Bài 3: PASS")

Bài 3: PASS


In [37]:
# Đáp án Bài 4
def answer_find_words(board, words):
    root = TrieNodeW()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNodeW())
        node.word = w

    m, n = len(board), len(board[0])
    result = []

    def dfs(r, c, parent):
        ch = board[r][c]
        child = parent.children.get(ch)
        if child is None:
            return
        if child.word is not None:
            result.append(child.word)
            child.word = None
        board[r][c] = "#"
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                dfs(nr, nc, child)
        board[r][c] = ch
        if not child.children and child.word is None:
            del parent.children[ch]

    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return result

assert sorted(answer_find_words(copy.deepcopy(board1), words1)) == ["eat", "oath"]
assert answer_find_words(copy.deepcopy(board2), words2) == []
assert sorted(answer_find_words([["a", "b"], ["b", "a"]], ["ab", "aba"])) == ["ab", "aba"]
print("Bài 4: PASS")

Bài 4: PASS


In [38]:
# Đáp án Bài 5 (#211)
class AnswerWordDictionary:
    def __init__(self):
        self.root = TrieNode()

    def addWord(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def search(self, word: str) -> bool:
        def dfs(i, node):
            if i == len(word):
                return node.is_end
            ch = word[i]
            if ch == ".":
                return any(dfs(i + 1, child) for child in node.children.values())
            child = node.children.get(ch)
            return child is not None and dfs(i + 1, child)
        return dfs(0, self.root)

def run_ops(ops, args):
    out, obj = [], None
    for op, arg in zip(ops, args):
        if op == "WordDictionary":
            obj = AnswerWordDictionary(); out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

got = run_ops(["WordDictionary","addWord","addWord","addWord","search","search","search","search"],
              [[],["bad"],["dad"],["mad"],["pad"],["bad"],[".ad"],["b.."]])
print(got)
assert got == [None, None, None, None, False, True, True, True]
print("Bài 5: PASS")

[None, None, None, None, False, True, True, True]
Bài 5: PASS


In [39]:
# Đáp án Bài 6
def answer_find_words_with_paths(board, words):
    root = build_trie_w(words)
    m, n = len(board), len(board[0])
    result, path = {}, []

    def dfs(r, c, parent):
        ch = board[r][c]
        child = parent.children.get(ch)
        if child is None:
            return
        path.append((r, c))
        if child.word is not None:
            result[child.word] = path[:]           # COPY
            child.word = None
        board[r][c] = "#"
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                dfs(nr, nc, child)
        board[r][c] = ch
        path.pop()
        if not child.children and child.word is None:
            del parent.children[ch]

    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return result

res = answer_find_words_with_paths(copy.deepcopy(board1), words1)
print(res)
assert sorted(res) == ["eat", "oath"]
assert all(is_valid_path(board1, w, p) for w, p in res.items())
show_board(board1, res["oath"])
print("Bài 6: PASS")

{'oath': [(0, 0), (0, 1), (1, 1), (2, 1)], 'eat': [(1, 3), (1, 2), (1, 1)]}
    c0 c1 c2 c3
r0  o1 a2 a  n 
r1  e  t3 a  e 
r2  i  h4 k  r 
r3  i  f  l  v 
Bài 6: PASS


## 23. Template nên nhớ

```python
class TrieNode:
    def __init__(self):
        self.children = {}
        self.word = None

class Solution:
    def findWords(self, board: List[List[str]], words: List[str]) -> List[str]:
        root = TrieNode()
        for w in words:                                   # 1. xây Trie
            node = root
            for ch in w:
                node = node.children.setdefault(ch, TrieNode())
            node.word = w

        m, n, result = len(board), len(board[0]), []

        def dfs(r, c, parent):
            ch = board[r][c]
            node = parent.children.get(ch)
            if node is None:                              # 2. cắt nhánh
                return
            if node.word:                                 # 3. ghi nhận + xóa
                result.append(node.word); node.word = None
            board[r][c] = "#"                             # 4. chọn
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    dfs(nr, nc, node)                     #    đi tiếp
            board[r][c] = ch                              #    bỏ chọn
            if not node.children and node.word is None:   # 5. tỉa lá
                del parent.children[ch]

        for r in range(m):
            for c in range(n):
                dfs(r, c, root)
        return result
```

Câu thần chú: **"Trie gom từ; DFS mang node; `get` None thì cắt; thấy từ thì lấy và xóa; `'#'` vào, chữ ra; lá rỗng thì tỉa."**

## 24. Checklist trước khi submit

- [ ] Xây Trie **một lần**, node cuối lưu `word` (không cần mang chuỗi theo DFS).
- [ ] Kiểm tra biên `0 <= nr < m and 0 <= nc < n` (đừng tin `board[-1]`).
- [ ] Đánh dấu `'#'` **trước** khi đi sang ô kề, trả lại chữ cũ ở **mọi** lối ra.
- [ ] Thấy từ thì **ghi nhận rồi đi tiếp** (không `return` sớm — `"oat"` / `"oath"`).
- [ ] Không trả trùng: `node.word = None` sau khi thêm (hoặc dùng `set`).
- [ ] Tỉa chỉ khi `not node.children and node.word is None`.
- [ ] Thử tay: Example 1 → `["eat","oath"]` (thứ tự tùy ý), Example 2 → `[]`, lưới `1x1`, lưới toàn `a`.